# Statistical Analysis of 30-Day Hospital Readmission
### Assessing Readmission Burden- Secondary Diagnosis, associated Factors, Statistial Significance and Effect Size.


## Purpose

This analysis examines the burden of 30-day hospital readmissions and evaluates whether selected factors differ between patients who were readmitted within 30 days and those who were not.

The analysis uses descriptive statistics and hypothesis testing to identify factors that show evidence of association with 30-day readmission. Effect-size measures are also calculated to distinguish statistically significant findings from associations that may have greater practical or analytical importance.

The resulting analysis table is used to support the Tableau Dashboard

## Business Questions

This analysis addresses four primary questions:
1. **Which continuous clinical and utilization variables differ between readmitted and non-readmitted patients?**
2. **Which categorical patient, admission, and hospital characteristics are associated with readmission?**
3. **Which statistically significant associations have meaningful effect sizes and may warrant further investigation or consideration in intervention planning?**

## Analytical Population

The analysis focuses on admissions with a **secondary diagnosis**, as defined in the prepared analytical dataset.

For clinical comparison analyses, patients with an outcome of 30-day readmission = 0 or 1 are included. Patients whose discharge status indicates death are excluded from analyses where comparison of post-discharge readmission risk is clinically inappropriate.

## Output

The final analytical dataset contains:

* Group-level descriptive statistics
* Statistical significance
* Percentage-point differences for categorical variables
* Cohen's d for continuous variables
* Cramér's V for categorical variables
* Direction of association
* Effect-size interpretation

The final table is exported for use in Tableau dashboard


## Statistical Methodology

### Analytical Approach

The analysis compares admissions with and without a 30-day readmission outcome using separate statistical methods for continuous and categorical variables.

### Continuous Variables

For continuous variables, the analysis reports:

* Mean among non-readmitted admissions
* Mean among readmitted admissions
* Mean difference
* Welch's independent-samples t-test
* p-value
* Cohen's d effect size
* Direction of the difference

Welch's t-test is used because it does not require the two groups to have equal variances.

### Categorical Variables

For categorical variables, the analysis reports:

* Distribution within the non-readmitted group
* Distribution within the readmitted group
* Percentage-point difference
* Chi-square test of independence
* p-value
* Cramér's V effect size
* Direction of the association

### Statistical Significance

A p-value below 0.05 is treated as statistically significant for the primary analysis.

Statistical significance is interpreted together with effect size rather than used as the sole criterion for identifying important factors.

### Effect Size

Effect sizes are included because large datasets can produce statistically significant results even when the practical difference is small.

For continuous variables:

# * |d| < 0.20 → Negligible
* |d| 0.20–<0.50 → Small
* |d| 0.50–<0.80 → Moderate
* |d| ≥ 0.80 → Large

For categorical variables:

* V < 0.10 → Negligible
* V 0.10–<0.30 → Small
* V 0.30–<0.50 → Moderate
* V ≥ 0.50 → Large

## Interpretation

These analyses identify **unadjusted associations** between individual variables and 30-day readmission.

They do not establish causality, and statistical significance does not imply clinical importance.

Variables identified here may be considered for further investigation, visualization, or inclusion in predictive modeling. Predictive modeling is performed separately using a modeling-specific dataset and methodology.


### Load Analytical Dataset

The analysis begins with the validated feature table prepared in BigQuery.

The table contains patient-level admission records and derived features used for downstream exploratory analysis, statistical analysis, Tableau visualization, and predictive modeling.

The statistical analysis does not perform raw-data cleaning or feature construction. Those activities were completed upstream in the SQL data-preparation workflow.


In [49]:
# ============================================================
# 1. LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from scipy import stats
from statsmodels.stats.multitest import multipletests

from google.colab import auth
from google.cloud import bigquery
from google.colab import files

In [50]:
# ============================================================
# 2. LOAD ANALYTICAL DATASET FROM BIGQUERY
# ============================================================

auth.authenticate_user()

client = bigquery.Client(
    project="healthcare-readmission-505214"
)

query = """
SELECT *
FROM `healthcare-readmission-505214.staging.fea_readmission_tableauV2`
"""

df = client.query(query).to_dataframe()

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

df.head()

Rows: 120,000
Columns: 79


,admission_id,patient_id,age,gender,patient_state,bpl_card,insurance_type,comorbidity_count,prev_admissions,admit_date,...,has_K70,has_P22,has_A09,has_A15,has_J45,has_S06,has_N39,has_S72,has_C50,has_A41
0,769a7a15-d6c8-437c-89e9-f74441f146d0,68c06f5e-6252-48d2-a9e7-557b790536f7,80,M,Telangana,True,Ayushman,3,2,2016-12-15,...,0,0,1,0,0,0,0,0,0,0
1,865b69f7-6dac-473f-a734-d73b829840f1,95cd61e5-3c8b-4602-a6f8-361d27e0bad1,7,M,Karnataka,True,Private,0,0,2023-07-23,...,0,0,1,0,0,0,0,0,0,0
2,7ab5c658-7805-4af7-8b9b-eced235a672f,d42c162b-38fe-4da7-9026-7f9cfb81cedd,40,F,Telangana,False,Ayushman,0,1,2017-09-03,...,0,0,0,0,0,0,0,0,0,1
3,67ad7851-3319-410f-b7a6-709ebab6e06d,2e47deb0-4a16-4f03-b6dc-5c46fe9e85fd,48,M,Maharashtra,True,None,0,0,2021-11-24,...,0,0,0,0,0,0,0,0,0,1
4,a6aa6af1-c712-4bbd-924b-20023b4ca060,e8779fad-cfe6-41c2-8a1d-2dc1732171af,36,M,Gujarat,True,Ayushman,0,0,2021-10-08,...,0,0,0,0,0,0,0,0,0,1


In [51]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 79 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   admission_id               120000 non-null  object 
 1   patient_id                 120000 non-null  object 
 2   age                        120000 non-null  Int64  
 3   gender                     120000 non-null  object 
 4   patient_state              120000 non-null  object 
 5   bpl_card                   120000 non-null  boolean
 6   insurance_type             120000 non-null  object 
 7   comorbidity_count          120000 non-null  Int64  
 8   prev_admissions            120000 non-null  Int64  
 9   admit_date                 120000 non-null  dbdate 
 10  discharge_date             120000 non-null  dbdate 
 11  los_days                   120000 non-null  Int64  
 12  admit_type                 120000 non-null  object 
 13  ward_type                  12

In [52]:
# ============================================================
# 3. DEFINE DIAGNOSIS TYPE
# ============================================================

df["diag_type"] = np.where(
    df["secondary_diagnosis_count"] > 0,
    "Secondary",
    "Primary"
)

df["diag_type"].value_counts()

,count
diag_type,
Secondary,77128
Primary,42872


In [53]:
# ============================================================
# 4. DEFINE ANALYTICAL POPULATION
# ============================================================

analysis_df = df[
    (df["diag_type"] == "Secondary") &
    (df["readmitted_30d"].isin([0, 1])) &
    (df["discharge_type"] != "Expired")
].copy()

print(f"Original admissions: {len(df):,}")

print(
    f"Secondary-diagnosis admissions: "
    f"{(df['diag_type'] == 'Secondary').sum():,}"
)

print(
    f"Excluded expired admissions: "
    f"{(df['discharge_type'] == 'Expired').sum():,}"
)

print(
    f"Analytical population: "
    f"{len(analysis_df):,}"
)

Original admissions: 120,000
Secondary-diagnosis admissions: 77,128
Excluded expired admissions: 7,413
Analytical population: 72,532


In [54]:
# ============================================================
# 5. READMISSION OUTCOME CHECK
# ============================================================

readmission_counts = (
    analysis_df["readmitted_30d"]
    .value_counts()
    .sort_index()
)

print("30-day readmission counts:")
print(readmission_counts)

30-day readmission counts:
readmitted_30d
0    60948
1    11584
Name: count, dtype: Int64


In [55]:
# ============================================================
# 6. READMISSION BURDEN
# ============================================================

total_admissions = len(analysis_df)

readmitted = (
    analysis_df["readmitted_30d"] == 1
).sum()

not_readmitted = (
    analysis_df["readmitted_30d"] == 0
).sum()

readmission_rate = (
    readmitted / total_admissions * 100
)

print(f"Total analytical admissions: {total_admissions:,}")
print(f"Readmitted within 30 days: {readmitted:,}")
print(f"Not readmitted: {not_readmitted:,}")
print(f"30-day readmission rate: {readmission_rate:.2f}%")

Total analytical admissions: 72,532
Readmitted within 30 days: 11,584
Not readmitted: 60,948
30-day readmission rate: 15.97%


In [56]:
# ============================================================
# 7. VARIABLE DEFINITIONS
# ============================================================

continuous_vars = [
    "age",
    "comorbidity_count",
    "prev_admissions",
    "los_days",
    "num_procedures",
    "charlson_index",
    "hba1c",
    "creatinine",
    "haemoglobin",
    "systolic_bp",
    "diagnosis_count"
]

categorical_vars = [
    "gender",
    "patient_state",
    "insurance_type",
    "admit_type",
    "ward_type",
    "discharge_type",
    "hospital_name",
    "hospital_state",
    "tier",
    "teaching",
    "bpl_card",
    "primary_diag_category"
]

In [57]:
# ============================================================
# 8. TABLEAU-FRIENDLY VARIABLE NAMES
# ============================================================

variable_labels = {

    # Continuous
    "age": "Age",
    "comorbidity_count": "Comorbidity count",
    "prev_admissions": "Previous admissions",
    "los_days": "Length of stay",
    "num_procedures": "Procedures",
    "charlson_index": "Charlson index",
    "hba1c": "HbA1c",
    "creatinine": "Creatinine",
    "haemoglobin": "Haemoglobin",
    "systolic_bp": "Systolic blood pressure",
    "diagnosis_count": "Diagnosis count",

    # Categorical
    "gender": "Gender",
    "patient_state": "Patient state",
    "insurance_type": "Insurance type",
    "admit_type": "Admission type",
    "ward_type": "Ward type",
    "discharge_type": "Discharge type",
    "hospital_name": "Hospital",
    "hospital_state": "Hospital state",
    "tier": "Hospital tier",
    "teaching": "Teaching hospital",
    "bpl_card": "BPL card",
    "primary_diag_category": "Primary diagnosis category"
}

In [58]:
# ============================================================
# 9. UNITS
# ============================================================

units = {

    "age": "years",
    "comorbidity_count": "count",
    "prev_admissions": "count",
    "los_days": "days",
    "num_procedures": "count",
    "charlson_index": "score",
    "hba1c": "%",
    "creatinine": "mg/dL",
    "haemoglobin": "g/dL",
    "systolic_bp": "mmHg",
    "diagnosis_count": "count"
}

In [59]:
# ============================================================
# 10. EFFECT-SIZE INTERPRETATION
# ============================================================

def classify_cohens_d(d):
    """
    Interpret Cohen's d using conventional magnitude thresholds.
    Classification is intended for dashboard interpretation
    and should be considered alongside clinical context.
    """

    if pd.isna(d):
        return np.nan

    d_abs = abs(d)

    if d_abs >= 0.80:
        return "Large"
    elif d_abs >= 0.50:
        return "Moderate"
    elif d_abs >= 0.20:
        return "Small"
    else:
        return "Negligible"


def classify_cramers_v(v):
    """
    Interpret Cramér's V using conventional magnitude thresholds.
    Classification is intended for dashboard interpretation
    and should be considered alongside context.
    """

    if pd.isna(v):
        return np.nan

    v_abs = abs(v)

    if v_abs >= 0.50:
        return "Large"
    elif v_abs >= 0.30:
        return "Moderate"
    elif v_abs >= 0.10:
        return "Small"
    else:
        return "Negligible"

In [60]:
# ============================================================
# 11. CONTINUOUS VARIABLE ANALYSIS
# ============================================================

continuous_rows = []

for var in continuous_vars:

    # --------------------------------------------------------
    # Split observations by readmission status
    # --------------------------------------------------------

    not_readmitted_values = (
        analysis_df.loc[
            analysis_df["readmitted_30d"] == 0,
            var
        ]
        .dropna()
    )

    readmitted_values = (
        analysis_df.loc[
            analysis_df["readmitted_30d"] == 1,
            var
        ]
        .dropna()
    )

    # --------------------------------------------------------
    # Sample sizes
    # --------------------------------------------------------

    n_not = len(not_readmitted_values)
    n_read = len(readmitted_values)

    # --------------------------------------------------------
    # Group means
    # --------------------------------------------------------

    mean_not = not_readmitted_values.mean()
    mean_read = readmitted_values.mean()

    # Difference = Readmitted - Not Readmitted
    difference = mean_read - mean_not

    # --------------------------------------------------------
    # Welch's independent-samples t-test
    # --------------------------------------------------------

    t_stat, p_value = stats.ttest_ind(
        not_readmitted_values,
        readmitted_values,
        equal_var=False,
        nan_policy="omit"
    )

    # --------------------------------------------------------
    # Cohen's d
    # --------------------------------------------------------

    sd_not = not_readmitted_values.std(ddof=1)
    sd_read = readmitted_values.std(ddof=1)

    if n_not > 1 and n_read > 1:

        pooled_sd = np.sqrt(
            (
                (n_not - 1) * sd_not**2 +
                (n_read - 1) * sd_read**2
            )
            /
            (n_not + n_read - 2)
        )

    else:
        pooled_sd = np.nan

    if pd.isna(pooled_sd) or pooled_sd == 0:
        cohens_d = np.nan
    else:
        cohens_d = difference / pooled_sd

    # --------------------------------------------------------
    # Direction
    # --------------------------------------------------------

    if difference > 0:
        direction = "Higher in readmitted"
    elif difference < 0:
        direction = "Lower in readmitted"
    else:
        direction = "No difference"

    # --------------------------------------------------------
    # Add result
    # --------------------------------------------------------

    continuous_rows.append({

        "DataType": "Continuous",
        "Variable": variable_labels.get(var, var),
        "Categorical": np.nan,

        "Not Readmitted": mean_not,
        "Readmitted": mean_read,
        "Difference": difference,

        "Unit": units.get(var, ""),

        "N Not Readmitted": n_not,
        "N Readmitted": n_read,

        "Test": "Welch's t-test",

        "Effect Size Type": "Cohen's d",
        "Effect Size": cohens_d,
        "Effect Size Interpretation":
            classify_cohens_d(cohens_d),

        "Direction": direction,

        "P-value": p_value
    })


continuous_table = pd.DataFrame(continuous_rows)

continuous_table

,DataType,Variable,Categorical,Not Readmitted,Readmitted,Difference,Unit,N Not Readmitted,N Readmitted,Test,Effect Size Type,Effect Size,Effect Size Interpretation,Direction,P-value
0,Continuous,Age,NaN,56.646666,62.796961,6.150295,years,60948,11584,Welch's t-test,Cohen's d,0.391782,Small,Higher in readmitted,0.000000e+00
1,Continuous,Comorbidity count,NaN,2.118101,2.855490,0.737390,count,60948,11584,Welch's t-test,Cohen's d,0.564452,Moderate,Higher in readmitted,0.000000e+00
2,Continuous,Previous admissions,NaN,1.106976,1.569233,0.462257,count,60948,11584,Welch's t-test,Cohen's d,0.395497,Small,Higher in readmitted,1.883977e-246
3,Continuous,Length of stay,NaN,6.526104,10.331319,3.805215,days,60948,11584,Welch's t-test,Cohen's d,0.675005,Moderate,Higher in readmitted,0.000000e+00
4,Continuous,Procedures,NaN,1.868609,2.460031,0.591422,count,60948,11584,Welch's t-test,Cohen's d,0.364918,Small,Higher in readmitted,1.442982e-221
5,Continuous,Charlson index,NaN,2.421408,3.195701,0.774293,score,60948,11584,Welch's t-test,Cohen's d,0.509664,Moderate,Higher in readmitted,0.000000e+00
6,Continuous,HbA1c,NaN,5.883885,5.822721,-0.061164,%,60948,11584,Welch's t-test,Cohen's d,-0.057191,Negligible,Lower in readmitted,2.001942e-09
7,Continuous,Creatinine,NaN,1.078571,1.134487,0.055916,mg/dL,60948,11584,Welch's t-test,Cohen's d,0.058881,Negligible,Higher in readmitted,2.167692e-07
8,Continuous,Haemoglobin,NaN,11.478938,10.779290,-0.699647,g/dL,60948,11584,Welch's t-test,Cohen's d,-0.330622,Small,Lower in readmitted,1.795173e-223
9,Continuous,Systolic blood pressure,NaN,137.117018,141.621547,4.504529,mmHg,60948,11584,Welch's t-test,Cohen's d,0.218281,Small,Higher in readmitted,1.016940e-97


In [61]:
# ============================================================
# 12. CATEGORICAL VARIABLE ANALYSIS
# ============================================================

categorical_rows = []

for var in categorical_vars:

    # --------------------------------------------------------
    # Remove missing values for the variable being tested
    # --------------------------------------------------------

    temp = (
        analysis_df[
            [var, "readmitted_30d"]
        ]
        .dropna()
        .copy()
    )

    # --------------------------------------------------------
    # Create contingency table
    # Rows = categories
    # Columns = readmission status
    # --------------------------------------------------------

    contingency = pd.crosstab(
        temp[var],
        temp["readmitted_30d"]
    )

    # Make sure both outcome groups exist
    if 0 not in contingency.columns:
        contingency[0] = 0

    if 1 not in contingency.columns:
        contingency[1] = 0

    contingency = contingency[[0, 1]]

    # --------------------------------------------------------
    # Chi-square test
    # --------------------------------------------------------

    chi2, p_value, dof, expected = stats.chi2_contingency(
        contingency
    )

    # --------------------------------------------------------
    # Expected-frequency quality check
    # --------------------------------------------------------

    expected_flat = expected.flatten()

    min_expected = expected_flat.min()

    pct_expected_below_5 = (
        (expected_flat < 5).mean() * 100
    )

    # --------------------------------------------------------
    # Cramér's V
    # --------------------------------------------------------

    n = contingency.to_numpy().sum()

    r, k = contingency.shape

    min_dim = min(r - 1, k - 1)

    if min_dim == 0 or n == 0:
        cramers_v = np.nan
    else:
        cramers_v = np.sqrt(
            chi2 / (n * min_dim)
        )

    effect_size = classify_cramers_v(cramers_v)

    # --------------------------------------------------------
    # Percentage distribution within each outcome group
    # --------------------------------------------------------

    pct = contingency.div(
        contingency.sum(axis=0),
        axis=1
    ) * 100

    # --------------------------------------------------------
    # Create one row per category
    # --------------------------------------------------------

    for category in contingency.index:

        not_pct = pct.loc[category, 0]
        read_pct = pct.loc[category, 1]

        # Difference = Readmitted % - Not Readmitted %
        difference = read_pct - not_pct

        if difference > 0:
            direction = "Higher in readmitted"
        elif difference < 0:
            direction = "Lower in readmitted"
        else:
            direction = "No difference"

        categorical_rows.append({

            "DataType": "Categorical",
            "Variable": variable_labels.get(var, var),
            "Categorical": category,

            "Not Readmitted": not_pct,
            "Readmitted": read_pct,
            "Difference": difference,

            "Unit": "percentage points",

            "N Not Readmitted": contingency.loc[category, 0],
            "N Readmitted": contingency.loc[category, 1],

            "Test": "Chi-square",

            "Effect Size Type": "Cramér's V",
            "Effect Size": cramers_v,
            "Effect Size Interpretation": effect_size,

            "Direction": direction,

            "P-value": p_value,

            "Minimum Expected Count": min_expected,
            "Percent Expected Counts <5":
                pct_expected_below_5
        })


categorical_table = pd.DataFrame(categorical_rows)

categorical_table.head(30)

,DataType,Variable,Categorical,Not Readmitted,Readmitted,Difference,Unit,N Not Readmitted,N Readmitted,Test,Effect Size Type,Effect Size,Effect Size Interpretation,Direction,P-value,Minimum Expected Count,Percent Expected Counts <5
0,Categorical,Gender,F,47.558575,47.056285,-0.502290,percentage points,28986,5451,Chi-square,Cramér's V,0.003742,Negligible,Lower in readmitted,6.018191e-01,119.302487,0.0
1,Categorical,Gender,M,51.415961,51.890539,0.474578,percentage points,31337,6011,Chi-square,Cramér's V,0.003742,Negligible,Higher in readmitted,6.018191e-01,119.302487,0.0
2,Categorical,Gender,Other,1.025464,1.053177,0.027712,percentage points,625,122,Chi-square,Cramér's V,0.003742,Negligible,Higher in readmitted,6.018191e-01,119.302487,0.0
3,Categorical,Patient state,Andhra Pradesh,6.725405,6.155041,-0.570364,percentage points,4099,713,Chi-square,Cramér's V,0.013283,Negligible,Lower in readmitted,5.425475e-01,386.814758,0.0
4,Categorical,Patient state,Bihar,3.352038,3.271754,-0.080284,percentage points,2043,379,Chi-square,Cramér's V,0.013283,Negligible,Lower in readmitted,5.425475e-01,386.814758,0.0
5,Categorical,Patient state,Gujarat,6.868150,6.750691,-0.117459,percentage points,4186,782,Chi-square,Cramér's V,0.013283,Negligible,Lower in readmitted,5.425475e-01,386.814758,0.0
6,Categorical,Patient state,Haryana,4.809018,4.730663,-0.078355,percentage points,2931,548,Chi-square,Cramér's V,0.013283,Negligible,Lower in readmitted,5.425475e-01,386.814758,0.0
7,Categorical,Patient state,Karnataka,8.387478,8.347721,-0.039757,percentage points,5112,967,Chi-square,Cramér's V,0.013283,Negligible,Lower in readmitted,5.425475e-01,386.814758,0.0
8,Categorical,Patient state,Kerala,5.809871,5.792472,-0.017398,percentage points,3541,671,Chi-square,Cramér's V,0.013283,Negligible,Lower in readmitted,5.425475e-01,386.814758,0.0
9,Categorical,Patient state,Madhya Pradesh,4.866444,5.326312,0.459869,percentage points,2966,617,Chi-square,Cramér's V,0.013283,Negligible,Higher in readmitted,5.425475e-01,386.814758,0.0


In [62]:
# ============================================================
# 13. MULTIPLE-COMPARISON ADJUSTMENT
#     Benjamini-Hochberg False Discovery Rate (FDR)
# ============================================================

# ------------------------------------------------------------
# Continuous variables
# Each variable represents one hypothesis test
# ------------------------------------------------------------

continuous_table["Adjusted P-value"] = multipletests(
    continuous_table["P-value"],
    method="fdr_bh"
)[1]

continuous_table["Adjusted Significant"] = np.where(
    continuous_table["Adjusted P-value"] < 0.05,
    "Yes",
    "No"
)


# ------------------------------------------------------------
# Categorical variables
# Adjust one p-value per categorical variable
# ------------------------------------------------------------

categorical_pvalues = (
    categorical_table[
        ["Variable", "P-value"]
    ]
    .drop_duplicates(subset="Variable")
    .copy()
)

categorical_pvalues["Adjusted P-value"] = multipletests(
    categorical_pvalues["P-value"],
    method="fdr_bh"
)[1]

categorical_pvalues["Adjusted Significant"] = np.where(
    categorical_pvalues["Adjusted P-value"] < 0.05,
    "Yes",
    "No"
)


# ------------------------------------------------------------
# Merge adjusted results back onto category rows
# ------------------------------------------------------------

categorical_table = categorical_table.drop(
    columns=[
        "Adjusted P-value",
        "Adjusted Significant"
    ],
    errors="ignore"
)

categorical_table = categorical_table.merge(
    categorical_pvalues,
    on=["Variable", "P-value"],
    how="left"
)

In [63]:
# ============================================================
# 14. COMBINE ANALYSIS RESULTS
# ============================================================

clinical_analysis_table = pd.concat(
    [
        continuous_table,
        categorical_table
    ],
    ignore_index=True
)

print(
    f"Combined analytical table: "
    f"{clinical_analysis_table.shape[0]:,} rows × "
    f"{clinical_analysis_table.shape[1]:,} columns"
)

Combined analytical table: 109 rows × 19 columns


In [64]:
# ============================================================
# 15. ORDER VARIABLES
# ============================================================

variable_order = (
    [variable_labels[v] for v in continuous_vars]
    +
    [variable_labels[v] for v in categorical_vars]
)

clinical_analysis_table["Variable"] = pd.Categorical(
    clinical_analysis_table["Variable"],
    categories=variable_order,
    ordered=True
)

clinical_analysis_table = (
    clinical_analysis_table
    .sort_values(
        ["DataType", "Variable", "Categorical"],
        na_position="first"
    )
    .reset_index(drop=True)
)

In [65]:
# ============================================================
# 16. CREATE DISPLAY TABLE
# ============================================================

display_table = clinical_analysis_table.copy()

display_table["Not Readmitted"] = (
    display_table["Not Readmitted"].round(2)
)

display_table["Readmitted"] = (
    display_table["Readmitted"].round(2)
)

display_table["Difference"] = (
    display_table["Difference"].round(2)
)

display_table["Effect Size"] = (
    display_table["Effect Size"].round(3)
)

display_table["P-value"] = (
    display_table["P-value"].round(6)
)

display_table["Adjusted P-value"] = (
    display_table["Adjusted P-value"].round(6)
)

display_table.head(30)

,DataType,Variable,Categorical,Not Readmitted,Readmitted,Difference,Unit,N Not Readmitted,N Readmitted,Test,Effect Size Type,Effect Size,Effect Size Interpretation,Direction,P-value,Adjusted P-value,Adjusted Significant,Minimum Expected Count,Percent Expected Counts <5
0,Categorical,Gender,F,47.56,47.06,-0.50,percentage points,28986,5451,Chi-square,Cramér's V,0.004,Negligible,Lower in readmitted,0.601819,0.656530,No,119.302487,0.0
1,Categorical,Gender,M,51.42,51.89,0.47,percentage points,31337,6011,Chi-square,Cramér's V,0.004,Negligible,Higher in readmitted,0.601819,0.656530,No,119.302487,0.0
2,Categorical,Gender,Other,1.03,1.05,0.03,percentage points,625,122,Chi-square,Cramér's V,0.004,Negligible,Higher in readmitted,0.601819,0.656530,No,119.302487,0.0
3,Categorical,Patient state,Andhra Pradesh,6.73,6.16,-0.57,percentage points,4099,713,Chi-square,Cramér's V,0.013,Negligible,Lower in readmitted,0.542547,0.651057,No,386.814758,0.0
4,Categorical,Patient state,Bihar,3.35,3.27,-0.08,percentage points,2043,379,Chi-square,Cramér's V,0.013,Negligible,Lower in readmitted,0.542547,0.651057,No,386.814758,0.0
5,Categorical,Patient state,Gujarat,6.87,6.75,-0.12,percentage points,4186,782,Chi-square,Cramér's V,0.013,Negligible,Lower in readmitted,0.542547,0.651057,No,386.814758,0.0
6,Categorical,Patient state,Haryana,4.81,4.73,-0.08,percentage points,2931,548,Chi-square,Cramér's V,0.013,Negligible,Lower in readmitted,0.542547,0.651057,No,386.814758,0.0
7,Categorical,Patient state,Karnataka,8.39,8.35,-0.04,percentage points,5112,967,Chi-square,Cramér's V,0.013,Negligible,Lower in readmitted,0.542547,0.651057,No,386.814758,0.0
8,Categorical,Patient state,Kerala,5.81,5.79,-0.02,percentage points,3541,671,Chi-square,Cramér's V,0.013,Negligible,Lower in readmitted,0.542547,0.651057,No,386.814758,0.0
9,Categorical,Patient state,Madhya Pradesh,4.87,5.33,0.46,percentage points,2966,617,Chi-square,Cramér's V,0.013,Negligible,Higher in readmitted,0.542547,0.651057,No,386.814758,0.0


In [66]:
# ============================================================
# 17. CONTINUOUS RESULTS
# ============================================================

continuous_display = display_table[
    display_table["DataType"] == "Continuous"
][
    [
        "Variable",
        "Not Readmitted",
        "Readmitted",
        "Difference",
        "Unit",
        "Test",
        "Effect Size Type",
        "Effect Size",
        "Effect Size Interpretation",
        "Direction",
        "P-value",
        "Adjusted P-value",
        "Adjusted Significant"
    ]
]

display(continuous_display)

,Variable,Not Readmitted,Readmitted,Difference,Unit,Test,Effect Size Type,Effect Size,Effect Size Interpretation,Direction,P-value,Adjusted P-value,Adjusted Significant
98,Age,56.65,62.80,6.15,years,Welch's t-test,Cohen's d,0.392,Small,Higher in readmitted,0.0,0.0,Yes
99,Comorbidity count,2.12,2.86,0.74,count,Welch's t-test,Cohen's d,0.564,Moderate,Higher in readmitted,0.0,0.0,Yes
100,Previous admissions,1.11,1.57,0.46,count,Welch's t-test,Cohen's d,0.395,Small,Higher in readmitted,0.0,0.0,Yes
101,Length of stay,6.53,10.33,3.81,days,Welch's t-test,Cohen's d,0.675,Moderate,Higher in readmitted,0.0,0.0,Yes
102,Procedures,1.87,2.46,0.59,count,Welch's t-test,Cohen's d,0.365,Small,Higher in readmitted,0.0,0.0,Yes
103,Charlson index,2.42,3.20,0.77,score,Welch's t-test,Cohen's d,0.510,Moderate,Higher in readmitted,0.0,0.0,Yes
104,HbA1c,5.88,5.82,-0.06,%,Welch's t-test,Cohen's d,-0.057,Negligible,Lower in readmitted,0.0,0.0,Yes
105,Creatinine,1.08,1.13,0.06,mg/dL,Welch's t-test,Cohen's d,0.059,Negligible,Higher in readmitted,0.0,0.0,Yes
106,Haemoglobin,11.48,10.78,-0.70,g/dL,Welch's t-test,Cohen's d,-0.331,Small,Lower in readmitted,0.0,0.0,Yes
107,Systolic blood pressure,137.12,141.62,4.50,mmHg,Welch's t-test,Cohen's d,0.218,Small,Higher in readmitted,0.0,0.0,Yes


In [67]:
# ============================================================
# 18. CATEGORICAL RESULTS
# ============================================================

categorical_display = display_table[
    display_table["DataType"] == "Categorical"
][
    [
        "Variable",
        "Categorical",
        "Not Readmitted",
        "Readmitted",
        "Difference",
        "Test",
        "Effect Size Type",
        "Effect Size",
        "Effect Size Interpretation",
        "Direction",
        "P-value",
        "Adjusted P-value",
        "Adjusted Significant"
    ]
]

display(categorical_display.head(50))

,Variable,Categorical,Not Readmitted,Readmitted,Difference,Test,Effect Size Type,Effect Size,Effect Size Interpretation,Direction,P-value,Adjusted P-value,Adjusted Significant
0,Gender,F,47.56,47.06,-0.50,Chi-square,Cramér's V,0.004,Negligible,Lower in readmitted,0.601819,0.656530,No
1,Gender,M,51.42,51.89,0.47,Chi-square,Cramér's V,0.004,Negligible,Higher in readmitted,0.601819,0.656530,No
2,Gender,Other,1.03,1.05,0.03,Chi-square,Cramér's V,0.004,Negligible,Higher in readmitted,0.601819,0.656530,No
3,Patient state,Andhra Pradesh,6.73,6.16,-0.57,Chi-square,Cramér's V,0.013,Negligible,Lower in readmitted,0.542547,0.651057,No
4,Patient state,Bihar,3.35,3.27,-0.08,Chi-square,Cramér's V,0.013,Negligible,Lower in readmitted,0.542547,0.651057,No
5,Patient state,Gujarat,6.87,6.75,-0.12,Chi-square,Cramér's V,0.013,Negligible,Lower in readmitted,0.542547,0.651057,No
6,Patient state,Haryana,4.81,4.73,-0.08,Chi-square,Cramér's V,0.013,Negligible,Lower in readmitted,0.542547,0.651057,No
7,Patient state,Karnataka,8.39,8.35,-0.04,Chi-square,Cramér's V,0.013,Negligible,Lower in readmitted,0.542547,0.651057,No
8,Patient state,Kerala,5.81,5.79,-0.02,Chi-square,Cramér's V,0.013,Negligible,Lower in readmitted,0.542547,0.651057,No
9,Patient state,Madhya Pradesh,4.87,5.33,0.46,Chi-square,Cramér's V,0.013,Negligible,Higher in readmitted,0.542547,0.651057,No


In [68]:
# ============================================================
# 19. VARIABLE-LEVEL STATISTICAL SUMMARY
# ============================================================

continuous_summary = continuous_table[
    [
        "Variable",
        "Test",
        "Effect Size Type",
        "Effect Size",
        "Effect Size Interpretation",
        "P-value",
        "Adjusted P-value",
        "Adjusted Significant"
    ]
].copy()


categorical_summary = categorical_pvalues.merge(
    categorical_table[
        [
            "Variable",
            "Test",
            "Effect Size Type",
            "Effect Size",
            "Effect Size Interpretation"
        ]
    ].drop_duplicates("Variable"),
    on="Variable",
    how="left"
)[
    [
        "Variable",
        "Test",
        "Effect Size Type",
        "Effect Size",
        "Effect Size Interpretation",
        "P-value",
        "Adjusted P-value",
        "Adjusted Significant"
    ]
]


statistical_summary = pd.concat(
    [
        continuous_summary,
        categorical_summary
    ],
    ignore_index=True
)

statistical_summary

,Variable,Test,Effect Size Type,Effect Size,Effect Size Interpretation,P-value,Adjusted P-value,Adjusted Significant
0,Age,Welch's t-test,Cohen's d,0.391782,Small,0.000000e+00,0.000000e+00,Yes
1,Comorbidity count,Welch's t-test,Cohen's d,0.564452,Moderate,0.000000e+00,0.000000e+00,Yes
2,Previous admissions,Welch's t-test,Cohen's d,0.395497,Small,1.883977e-246,3.453959e-246,Yes
3,Length of stay,Welch's t-test,Cohen's d,0.675005,Moderate,0.000000e+00,0.000000e+00,Yes
4,Procedures,Welch's t-test,Cohen's d,0.364918,Small,1.442982e-221,1.984100e-221,Yes
5,Charlson index,Welch's t-test,Cohen's d,0.509664,Moderate,0.000000e+00,0.000000e+00,Yes
6,HbA1c,Welch's t-test,Cohen's d,-0.057191,Negligible,2.001942e-09,2.202137e-09,Yes
7,Creatinine,Welch's t-test,Cohen's d,0.058881,Negligible,2.167692e-07,2.167692e-07,Yes
8,Haemoglobin,Welch's t-test,Cohen's d,-0.330622,Small,1.795173e-223,2.820987e-223,Yes
9,Systolic blood pressure,Welch's t-test,Cohen's d,0.218281,Small,1.016940e-97,1.242927e-97,Yes


In [69]:
# ============================================================
# 20. STATISTICAL SIGNIFICANCE SUMMARY
# ============================================================

print("Nominal significance (p < 0.05)")
print("--------------------------------")

print(
    "Continuous variables:",
    (continuous_table["P-value"] < 0.05).sum(),
    "of",
    len(continuous_table)
)

print(
    "Categorical variables:",
    (
        categorical_pvalues["P-value"] < 0.05
    ).sum(),
    "of",
    len(categorical_pvalues)
)


print("\nFDR-adjusted significance")
print("--------------------------------")

print(
    "Continuous variables:",
    (
        continuous_table["Adjusted P-value"] < 0.05
    ).sum(),
    "of",
    len(continuous_table)
)

print(
    "Categorical variables:",
    (
        categorical_pvalues["Adjusted P-value"] < 0.05
    ).sum(),
    "of",
    len(categorical_pvalues)
)

Nominal significance (p < 0.05)
--------------------------------
Continuous variables: 11 of 11
Categorical variables: 6 of 12

FDR-adjusted significance
--------------------------------
Continuous variables: 11 of 11
Categorical variables: 6 of 12


In [70]:
# ============================================================
# 21. CHI-SQUARE ASSUMPTION CHECK
# ============================================================

chi_square_quality = (
    categorical_table[
        [
            "Variable",
            "Minimum Expected Count",
            "Percent Expected Counts <5"
        ]
    ]
    .drop_duplicates("Variable")
    .copy()
)

chi_square_quality["Expected Count Flag"] = np.where(
    chi_square_quality["Percent Expected Counts <5"] > 20,
    "Review",
    "Adequate"
)

display(chi_square_quality)

,Variable,Minimum Expected Count,Percent Expected Counts <5,Expected Count Flag
0,Gender,119.302487,0.0,Adequate
3,Patient state,386.814758,0.0,Adequate
18,Insurance type,1936.948547,0.0,Adequate
22,Admission type,1497.270170,0.0,Adequate
25,Ward type,430.255556,0.0,Adequate
29,Discharge type,996.423317,0.0,Adequate
32,Hospital,330.916671,0.0,Adequate
65,Hospital state,345.609883,0.0,Adequate
80,Hospital tier,2808.639276,0.0,Adequate
83,Teaching hospital,4912.962665,0.0,Adequate


In [71]:
# ============================================================
# 22. EXPORT TABLEAU-READY ANALYTICAL DATASET
# ============================================================

output_file = "clinical_readmission_analysis.csv"

clinical_analysis_table.to_csv(
    output_file,
    index=False
)

print(f"Saved: {output_file}")

Saved: clinical_readmission_analysis.csv


In [72]:
# ============================================================
# 23. DOWNLOAD CSV
# ============================================================

files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [73]:
# ============================================================
# 24. EXPORT VARIABLE-LEVEL SUMMARY
# ============================================================

summary_file = "clinical_readmission_statistical_summary.csv"

statistical_summary.to_csv(
    summary_file,
    index=False
)

print(f"Saved: {summary_file}")

files.download(summary_file)

Saved: clinical_readmission_statistical_summary.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Interpretation and Limitations

This analysis identifies **unadjusted statistical associations** between selected patient, clinical, admission, insurance, hospital, and diagnosis characteristics and 30-day hospital readmission among the defined analytical population.

Statistical significance indicates evidence that an observed difference or association is unlikely to be explained by sampling variation alone under the statistical test assumptions. It does not establish causality or indicate that a variable is clinically important.

Effect sizes are therefore reported alongside p-values to provide additional context regarding the magnitude of observed differences or associations.

Because multiple variables are tested, Benjamini-Hochberg false discovery rate (FDR) adjustment is applied to the statistical results. Both nominal and FDR-adjusted significance should be considered when interpreting the findings.

The results should be viewed as a screening and analytical assessment rather than evidence of causal relationships. Variables identified through this analysis may warrant further investigation, visualization, clinical review, or consideration as candidate predictors in the separate predictive modeling workflow.

### Key Limitations

* The analyses are unadjusted and do not control for potential confounding variables.
* Statistical association does not establish causation.
* Statistical significance does not necessarily indicate clinical or operational importance.
* Multiple comparisons increase the possibility of false-positive findings; FDR adjustment is used to address this concern.
* Categorical variables with sparse expected cell counts require cautious interpretation of Chi-square results.
* The analysis population is restricted to secondary-diagnosis admissions and excludes expired admissions.
* Findings are based on the available analytical dataset and may not generalize beyond the population represented by the data.
* Clinical interpretation and intervention planning should incorporate subject-matter expertise and, where appropriate, further validation.

### Relationship to Predictive Modeling

This notebook addresses the **statistical association** component of the project.

The separate predictive modeling workflow addresses a different question: whether available patient and admission characteristics can be used to identify patients at higher predicted risk of 30-day readmission.

Therefore:

**Statistical analysis asks:**
*Which variables show evidence of association with readmission, and how large are the observed differences or associations?*

**Predictive modeling asks:**
*How well can available information identify patients at higher predicted risk of readmission?*

The two analyses are complementary but serve different purposes.
